# Bio-Sniff QFF 2026 Phase 2 — PS-O-OI

This notebook uses the locked Phase 1-derived instances in `data/instance.json` and the official helpers in `qff_phase2.py`. The synthetic scores are not field measurements. Cell 2 performs ideal statevector angle optimization and routing diagnostics only; it does not run Bell experiments, noisy benchmarks, or the penalty sweep.

## Cell 1 — Environment, Dependencies, and Reproducibility

This cell verifies the supported Python range and required packages, imports the organizer helpers, creates output folders, and records the official settings used below.

In [1]:
import importlib.metadata as package_metadata
import inspect
import json
import sys
import time
from pathlib import Path

import numpy as np
import scipy
from IPython.display import Markdown, display
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from scipy.optimize import minimize

import qff_phase2 as qff


# -----------------------------------------------------------------------------
# Environment and required project files
# -----------------------------------------------------------------------------
if not ((3, 10) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError(
        f"Python {sys.version_info.major}.{sys.version_info.minor} is unsupported; "
        "use Python 3.10, 3.11, or 3.12."
    )

ROOT = Path.cwd()
if not (ROOT / "qff_phase2.py").is_file():
    raise FileNotFoundError("Run this notebook from the Phase 2 project root.")

INSTANCE_PATH = ROOT / "data" / "instance.json"
PROCESSOR_PATHS = {
    "B": ROOT / "processors" / "processor_B.json",
    "D": ROOT / "processors" / "processor_D.json",
}
for required_path in (INSTANCE_PATH, *PROCESSOR_PATHS.values()):
    if not required_path.is_file():
        raise FileNotFoundError(f"Required project file is missing: {required_path}")

PACKAGE_NAMES = {
    "qiskit": "qiskit",
    "qiskit-aer": "qiskit-aer",
    "numpy": "numpy",
    "scipy": "scipy",
    "networkx": "networkx",
    "matplotlib": "matplotlib",
    "ipython": "ipython",
}
PACKAGE_VERSIONS = {}
for distribution_name in PACKAGE_NAMES:
    try:
        PACKAGE_VERSIONS[distribution_name] = package_metadata.version(distribution_name)
    except package_metadata.PackageNotFoundError as error:
        raise RuntimeError(
            f"Required package {distribution_name!r} is missing. "
            "Install the project requirements into this notebook's Python environment."
        ) from error

REQUIRED_HELPERS = (
    "simulator",
    "transpile_for",
    "circuit_metrics",
    "counts_to_x",
    "load_processor",
    "validate_processor_d",
)
for helper_name in REQUIRED_HELPERS:
    if not callable(getattr(qff, helper_name, None)):
        raise RuntimeError(f"Official helper qff_phase2.{helper_name} is unavailable.")

print(f"Python: {sys.version.split()[0]} (supported)")
for package_name, version in PACKAGE_VERSIONS.items():
    print(f"{package_name}: {version}")
print("Official helper signatures:")
for helper_name in REQUIRED_HELPERS:
    print(f"  {helper_name}{inspect.signature(getattr(qff, helper_name))}")


# -----------------------------------------------------------------------------
# Output folders and reproducibility settings
# -----------------------------------------------------------------------------
RESULTS_DIR = ROOT / "results"
OPTIMIZATION_DIR = RESULTS_DIR / "optimization"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"
for output_dir in (RESULTS_DIR, OPTIMIZATION_DIR, FIGURES_DIR, TABLES_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)

QAOA_P = 1
INITIALIZATION_SEEDS = (0, 1, 2, 3, 4)
OFFICIAL_SETTINGS = {
    "basis_gates": list(qff.BASIS_GATES),
    "optimization_level": qff.OPT_LEVEL,
    "seed_transpiler_and_simulator": qff.SEED,
    "final_evaluation_shots": qff.SHOTS,
    "p1_depolarizing": qff.P1_DEPOL,
    "p2_depolarizing_cx": qff.P2_DEPOL,
    "readout_bitflip": qff.P_READOUT,
}
OPTIMIZER_NAME = "COBYLA"
OPTIMIZER_MAXITER = 60
SMOKE_MAXITER = 8
MAX_ESTIMATED_FIVE_SEED_SECONDS = 120.0
RECOMPUTE_IDEAL_OPTIMIZATION = False
OPTIMIZATION_CACHE_PATH = OPTIMIZATION_DIR / "ideal_qaoa_optimization.json"

print("QAOA depth:", QAOA_P)
print("Initialization seeds:", INITIALIZATION_SEEDS)
print("Optimizer:", OPTIMIZER_NAME, "maxiter:", OPTIMIZER_MAXITER)
print("Official settings:", OFFICIAL_SETTINGS)
print("Optimization cache:", OPTIMIZATION_CACHE_PATH)
print("Setup complete; no experiments were run in Cell 1.")

Python: 3.12.10 (supported)
qiskit: 2.6.1
qiskit-aer: 0.18.2
numpy: 2.5.4
scipy: 1.18.1
networkx: 3.7
matplotlib: 3.11.2
ipython: 9.17.1
Official helper signatures:
  simulator(noisy: 'bool', processor: 'dict | None' = None, noise_model=None) -> 'AerSimulator'
  transpile_for(qc: 'QuantumCircuit', processor: 'dict') -> 'QuantumCircuit'
  circuit_metrics(qc: 'QuantumCircuit', processor: 'dict') -> 'dict'
  counts_to_x(bitstring: 'str') -> 'tuple'
  load_processor(path: 'str') -> 'dict'
  validate_processor_d(p: 'dict', required_qubits: 'int') -> 'list'
QAOA depth: 1
Initialization seeds: (0, 1, 2, 3, 4)
Optimizer: COBYLA maxiter: 60
Official settings: {'basis_gates': ['rz', 'sx', 'x', 'cx'], 'optimization_level': 1, 'seed_transpiler_and_simulator': 42, 'final_evaluation_shots': 4096, 'p1_depolarizing': 0.001, 'p2_depolarizing_cx': 0.005, 'readout_bitflip': 0.01}
Optimization cache: C:\Users\DELL-PC\OneDrive\Desktop\Bio-Sniff-QFF-2026-Phase-2\results\optimization\ideal_qaoa_optimization.

## Cell 2 — Locked-Instance QAOA Core

The implementation below enumerates the exact valid-placement landscape and uniform-random baselines, performs a short ideal smoke test first, then optimizes the p=1 angles from seeds 0–4 if the measured runtime estimate is acceptable. It displays the logical and processor-transpiled circuits and reports only routing metrics here; no Bell, noisy benchmark matrix, or penalty sweep is run.

In [2]:
from pathlib import Path

from src.biosniff_pipeline import run_pipeline


# Cell 1 defines ROOT and verifies the supported environment; this explicit
# guard makes a direct Cell 2 run fail with a useful message instead of NameError.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "qff_phase2.py").is_file():
    raise FileNotFoundError("Run Cell 1 first from the Phase 2 project root.")

PIPELINE_RESULTS = run_pipeline(PROJECT_ROOT)
MAIN_CIRCUITS = PIPELINE_RESULTS["main_circuits"]

print("\nLogical p=1 circuit for Instance L:")
display(MAIN_CIRCUITS["logical"].draw(output="text", fold=100))
for processor_label in ("B", "D"):
    processor = qff.load_processor(str(PROJECT_ROOT / "processors" / f"processor_{processor_label}.json"))
    print(f"\nProcessor {processor_label} transpiled L circuit:")
    print("Official circuit metrics:", qff.circuit_metrics(MAIN_CIRCUITS["logical"], processor))
    display(MAIN_CIRCUITS[processor_label].draw(output="text", fold=100))

print("\nStep 2 pipeline complete. Bell, benchmark, penalty, report and cache outputs are in PIPELINE_RESULTS.")

Prior ideal optimization cache matches the locked instance and processor hashes.
Loaded 10 archived Step 1A seed attempts from ideal_qaoa_optimization.pre-end2end-20261030.bak.json.
L seed 0: best energy=-25.55801092555826; converged=True; attempts=2
L seed 1: best energy=-18.861175716863347; converged=True; attempts=1
L seed 2: best energy=-21.63173321072276; converged=True; attempts=2
L seed 3: best energy=-16.292007580418062; converged=True; attempts=2
L seed 4: best energy=-25.27731612679246; converged=True; attempts=2
S seed 0: best energy=-26.647556011879715; converged=True; attempts=1
S seed 1: best energy=-25.983617546408723; converged=True; attempts=1
S seed 2: best energy=-26.647556389101222; converged=True; attempts=1
S seed 3: best energy=-26.647555190669475; converged=True; attempts=1
S seed 4: best energy=-24.993259274271523; converged=True; attempts=1


Pipeline implementation hash changed; cached outputs will be reused only after record validation.
Loaded compatible Phase 2 experiment checkpoint cache.


Penalty P=0: QUBO minimum valid=False; energy=-9.64326462; tied minima=1


Penalty P=0.5: QUBO minimum valid=False; energy=-1.90468571; tied minima=1


Penalty P=1: QUBO minimum valid=True; energy=-1.45247143; tied minima=1


...


Processor B transpiled L circuit:
Official circuit metrics: {'processor': 'Processor B', 'executable': True, 'logical_qubits': 9, 'physical_qubits': 9, 'logical_cx': 72, 'cx': 121, 'one_q_gates': 108, 'depth': 117, 'two_q_depth': 79, 'swaps': 17, 'cx_overhead': 49}


global phase: 3π/4
         ┌─────────┐┌────┐┌────────────┐                                                          »
q_8 -> 0 ┤ Rz(π/2) ├┤ √X ├┤ Rz(2.7905) ├──────────────────────────────────────────────────────────»
         ├─────────┤├────┤├────────────┤                                                          »
q_4 -> 1 ┤ Rz(π/2) ├┤ √X ├┤ Rz(2.1041) ├──────────────────────────────────────────────────────────»
         ├─────────┤├────┤├────────────┤                                                          »
q_0 -> 2 ┤ Rz(π/2) ├┤ √X ├┤ Rz(2.5188) ├──■──────────────────■─────────■───────────────────────■──»
         ├─────────┤├────┤├────────────┤┌─┴─┐┌────────────┐┌─┴─┐┌───┐  │                ┌───┐  │  »
q_1 -> 3 ┤ Rz(π/2) ├┤ √X ├┤ Rz(2.4201) ├┤ X ├┤ Rz(6.6929) ├┤ X ├┤ X ├──┼────────■───────┤ X ├──┼──»
         ├─────────┤├────┤├────────────┤└───┘└────────────┘└───┘└─┬─┘  │      ┌─┴─┐     └─┬─┘  │  »
q_3 -> 4 ┤ Rz(π/2) ├┤ √X ├┤ Rz(2.4077) ├──────────────────────────■────┼──────┤ X ├───────■────┼──»
         ├─────────┤├────┤├────────────┤                             ┌─┴─┐┌───┴───┴────┐     ┌─┴─┐»
q_2 -> 5 ┤ Rz(π/2) ├┤ √X ├┤ Rz(2.4249) ├─────────────────────────────┤ X ├┤ Rz(6.5133) ├─────┤ X ├»
         ├─────────┤├────┤├────────────┤                             └───┘└────────────┘     └───┘»
q_5 -> 6 ┤ Rz(π/2) ├┤ √X ├┤ Rz(2.3452) ├──────────────────────────────────────────────────────────»
         ├─────────┤├────┤├───────────┬┘                                                          »
q_6 -> 7 ┤ Rz(π/2) ├┤ √X ├┤ Rz(2.679) ├───────────────────────────────────────────────────────────»
         ├─────────┤├────┤├───────────┴┐                                                          »
q_7 -> 8 ┤ Rz(π/2) ├┤ √X ├┤ Rz(2.3714) ├──────────────────────────────────────────────────────────»
         └─────────┘└────┘└────────────┘                                                          »
 meas: 9/═════════════════════════════════════════════════════════════════════════════════════════»
                                                                                                  »
«                                                                                                »
«q_8 -> 0 ───────────────────────────────────────────────────────────────────────────────────────»
«                                 ┌───┐┌────────────┐┌───┐                                       »
«q_4 -> 1 ────────────────────────┤ X ├┤ Rz(6.6125) ├┤ X ├───────────────────────────────────────»
«                                 └─┬─┘└────────────┘└─┬─┘                                       »
«q_0 -> 2 ──■──────────────────■────■──────────────────■─────────■───────────────────────■───────»
«         ┌─┴─┐┌────────────┐┌─┴─┐┌───┐┌────────────┐┌───┐       │                       │  ┌───┐»
«q_1 -> 3 ┤ X ├┤ Rz(6.6789) ├┤ X ├┤ X ├┤ Rz(6.5838) ├┤ X ├───────┼───────────────────────┼──┤ X ├»
«         └───┘└────────────┘└───┘└─┬─┘└────────────┘└─┬─┘┌───┐  │                ┌───┐  │  └─┬─┘»
«q_3 -> 4 ──■──────────────────■────■──────────────────■──┤ X ├──┼────────■───────┤ X ├──┼────■──»
«         ┌─┴─┐┌────────────┐┌─┴─┐                        └─┬─┘  │      ┌─┴─┐     └─┬─┘  │       »
«q_2 -> 5 ┤ X ├┤ Rz(6.6797) ├┤ X ├──────────────────────────■────┼──────┤ X ├───────■────┼───────»
«         └───┘└────────────┘└───┘                             ┌─┴─┐┌───┴───┴────┐     ┌─┴─┐     »
«q_5 -> 6 ─────────────────────────────────────────────────────┤ X ├┤ Rz(6.4365) ├─────┤ X ├─────»
«                                                              └───┘└────────────┘     └───┘     »
«q_6 -> 7 ───────────────────────────────────────────────────────────────────────────────────────»
«                                                                                                »
«q_7 -> 8 ───────────────────────────────────────────────────────────────────────────────────────»
«                                                                                                »



Processor D transpiled L circuit:
Official circuit metrics: {'processor': 'Processor D', 'executable': True, 'logical_qubits': 9, 'physical_qubits': 9, 'logical_cx': 72, 'cx': 102, 'one_q_gates': 108, 'depth': 87, 'two_q_depth': 59, 'swaps': 10, 'cx_overhead': 30}


global phase: 3π/4
         ┌─────────┐┌────┐┌────────────┐                                                ┌───┐»
q_3 -> 0 ┤ Rz(π/2) ├┤ √X ├┤ Rz(2.4077) ├────────────────────────────────────────────────┤ X ├»
         ├─────────┤├────┤├────────────┤                                                └─┬─┘»
q_0 -> 1 ┤ Rz(π/2) ├┤ √X ├┤ Rz(2.5188) ├──■──────────────────■────■──────────────────■────■──»
         ├─────────┤├────┤├────────────┤┌─┴─┐┌────────────┐┌─┴─┐  │                  │       »
q_1 -> 2 ┤ Rz(π/2) ├┤ √X ├┤ Rz(2.4201) ├┤ X ├┤ Rz(6.6929) ├┤ X ├──┼──────────────────┼────■──»
         ├─────────┤├────┤├────────────┤└───┘└────────────┘└───┘┌─┴─┐┌────────────┐┌─┴─┐┌─┴─┐»
q_2 -> 3 ┤ Rz(π/2) ├┤ √X ├┤ Rz(2.4249) ├────────────────────────┤ X ├┤ Rz(6.5133) ├┤ X ├┤ X ├»
         ├─────────┤├────┤├────────────┤                        └───┘└────────────┘└───┘└───┘»
q_8 -> 4 ┤ Rz(π/2) ├┤ √X ├┤ Rz(2.7905) ├─────────────────────────────────────────────────────»
         ├─────────┤├────┤├────────────┤                                                     »
q_7 -> 5 ┤ Rz(π/2) ├┤ √X ├┤ Rz(2.3714) ├─────────────────────────────────────────────────────»
         ├─────────┤├────┤├────────────┤                                                     »
q_4 -> 6 ┤ Rz(π/2) ├┤ √X ├┤ Rz(2.1041) ├─────────────────────────────────────────────────────»
         ├─────────┤├────┤├────────────┤                                                     »
q_5 -> 7 ┤ Rz(π/2) ├┤ √X ├┤ Rz(2.3452) ├─────────────────────────────────────────────────────»
         ├─────────┤├────┤├───────────┬┘                                                     »
q_6 -> 8 ┤ Rz(π/2) ├┤ √X ├┤ Rz(2.679) ├──────────────────────────────────────────────────────»
         └─────────┘└────┘└───────────┘                                                      »
 meas: 9/════════════════════════════════════════════════════════════════════════════════════»
                                                                                             »
«         ┌────────────┐┌───┐┌───┐┌────────────┐     ┌───┐                                  »
«q_3 -> 0 ┤ Rz(6.6789) ├┤ X ├┤ X ├┤ Rz(6.5838) ├─────┤ X ├──────────────────────────────────»
«         └────────────┘└─┬─┘└─┬─┘└───┬───┬────┘     └─┬─┘┌───┐                             »
«q_0 -> 1 ────────────────■────┼──────┤ X ├───────■────┼──┤ X ├─────────────────────────────»
«                              │      └─┬─┘       │    │  └─┬─┘                             »
«q_1 -> 2 ────────────────■────■────────┼─────────┼────■────┼───────────────────────────────»
«         ┌────────────┐┌─┴─┐           │         │         │                               »
«q_2 -> 3 ┤ Rz(6.6797) ├┤ X ├───────────┼─────────┼─────────┼───────────────────────────────»
«         └────────────┘└───┘           │         │         │                               »
«q_8 -> 4 ──────────────────────────────┼─────────┼─────────┼────────────────────────────■──»
«                                       │         │         │                            │  »
«q_7 -> 5 ──────────────────────────────┼─────────┼─────────┼────────────────────────────┼──»
«                                       │         │         │  ┌───┐┌────────────┐┌───┐┌─┴─┐»
«q_4 -> 6 ──────────────────────────────┼─────────┼─────────┼──┤ X ├┤ Rz(6.6125) ├┤ X ├┤ X ├»
«                                       │         │         │  └─┬─┘└────────────┘└─┬─┘├───┤»
«q_5 -> 7 ──────────────────────────────┼─────────┼─────────┼────┼──────────────────┼──┤ X ├»
«                                       │       ┌─┴─┐       │    │                  │  └─┬─┘»
«q_6 -> 8 ──────────────────────────────■───────┤ X ├───────■────■──────────────────■────■──»
«                                               └───┘                                       »
« meas: 9/══════════════════════════════════════════════════════════════════════════════════»
«                                                                                           »
«           


Step 2 pipeline complete. Bell, benchmark, penalty, report and cache outputs are in PIPELINE_RESULTS.
